In [13]:
TF_ENABLE_ONEDNN_OPTS=0

<h2>Section A: LLM Foundations & Hugging Face</h2>

In [2]:
import warnings
warnings.filterwarnings("ignore")

from transformers import pipeline, AutoTokenizer

2026-05-04 11:05:08.172270: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-04 11:05:08.184665: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777892708.199849     160 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777892708.204863     160 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777892708.217304     160 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

Use Hugging Face's transformers library to load DistilGPT-2 — a lighter, faster version of GPT-2 (~80 MB). The pipeline wrapper abstracts away the tokenization and inference steps, we can just pass in a string and get text back

In [3]:
generator = pipeline("text-generation", model="distilgpt2")

print("Model loaded!\n")

Model loaded!



The show() Helper
A simple pretty-printer that formats the prompt and its generated outputs cleanly. Nothing model-related — just presentation logic.

In [4]:
# ════════════════════════════════════════════════════════════
#  HELPER — pretty-print results
# ════════════════════════════════════════════════════════════
def show(title: str, prompt: str, results: list, notes: str = ""):
    print("─" * 62)
    print(f"  {title}")
    if notes:
        print(f"  ℹ️  {notes}")
    print("─" * 62)
    print(f"  Prompt : \"{prompt}\"")
    for i, r in enumerate(results, 1):
        label = f"  Seq {i}  : " if len(results) > 1 else "  Output : "
        print(f"{label}{r['generated_text']}")
    print()

Test 1 — Baseline
A simple generation with default settings. Establishes what the model produces without tuning anything.

In [5]:
show(
    title  = "TEST 1 │ Baseline Prompt",
    prompt = "AI is transforming industries by",
    results= generator(
        "AI is transforming industries by",
        max_length=40,
        num_return_sequences=1
    )
)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


──────────────────────────────────────────────────────────────
  TEST 1 │ Baseline Prompt
──────────────────────────────────────────────────────────────
  Prompt : "AI is transforming industries by"
  Output : AI is transforming industries by design, manufacturing, education, health, and environmental sectors to create an environment in which only one part of one species is a member of the population, by creating a natural habitat



Test 2 — Different Topic
Same parameters as Test 1, but a different prompt. Shows that the model's output direction is entirely steered by the input text.

In [6]:
# ════════════════════════════════════════════════════════════
#  TEST 2 · Different topic — observe domain shift in output
# ════════════════════════════════════════════════════════════
show(
    title  = "TEST 2 │ Different Topic Prompt",
    prompt = "The future of medicine depends on",
    results= generator(
        "The future of medicine depends on",
        max_length=40,
        num_return_sequences=1
    ),
    notes  = "Same params, different topic → output changes direction"
)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


──────────────────────────────────────────────────────────────
  TEST 2 │ Different Topic Prompt
  ℹ️  Same params, different topic → output changes direction
──────────────────────────────────────────────────────────────
  Prompt : "The future of medicine depends on"
  Output : The future of medicine depends on a variety of factors. The most difficult part about these matters is the nature of medicine. For one, the current age is probably the oldest of the 20th Century,



Test 3 — Varying max_length
Runs the same prompt twice with max_length=20 and max_length=60. Demonstrates that max_length controls the total token count (prompt + output), so longer values allow the model to elaborate further.

In [7]:
# ════════════════════════════════════════════════════════════
#  TEST 3 · Vary max_length → see how output truncates / grows
# ════════════════════════════════════════════════════════════
prompt3 = "Space exploration will lead to"
print("─" * 62)
print("  TEST 3 │ Varying max_length (20 vs 60 tokens)")
print("  ℹ️  Watch how the story grows as we allow more tokens")
print("─" * 62)
print(f"  Prompt : \"{prompt3}\"")

r_short = generator(prompt3, max_length=20,  num_return_sequences=1)
r_long  = generator(prompt3, max_length=60,  num_return_sequences=1)


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


──────────────────────────────────────────────────────────────
  TEST 3 │ Varying max_length (20 vs 60 tokens)
  ℹ️  Watch how the story grows as we allow more tokens
──────────────────────────────────────────────────────────────
  Prompt : "Space exploration will lead to"


In [8]:
print(f"  max=20  : {r_short[0]['generated_text']}")

  max=20  : Space exploration will lead to an eventual return of "the best that ever existed".


In [9]:
print(f"  max=60  : {r_long[0]['generated_text']}")

  max=60  : Space exploration will lead to significant investments in the space program," said NASA Administrator Jens Stoltenberg in a statement. "This is the first step toward a full-size mission, with the goal of establishing more affordable and cost-effective, science-driven space vehicles and instruments to replace aging


Test 4 — Multiple Sequences
Uses num_return_sequences=3 with do_sample=True and temperature=0.9. This generates three different completions from the same prompt in a single call, showing the natural diversity from sampling.

In [18]:
# ════════════════════════════════════════════════════════════
#  TEST 4 · Multiple sequences — diversity in one call
# ════════════════════════════════════════════════════════════
show(
    title  = "TEST 4 │ Multiple Sequences (num_return_sequences=3)",
    prompt = "Robots will soon be able to",
    results= generator(
        "Robots will soon be able to",
        max_length=45,
        num_return_sequences=3,
        do_sample=True,
        temperature=0.9
    ),
    notes  = "Three different continuations from the same prompt"
)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


──────────────────────────────────────────────────────────────
  TEST 4 │ Multiple Sequences (num_return_sequences=3)
  ℹ️  Three different continuations from the same prompt
──────────────────────────────────────────────────────────────
  Prompt : "Robots will soon be able to"
  Seq 1  : Robots will soon be able to make more of a virtual reality experience. There's even a game for PC, but what's better for you, like this:


This is not a new concept. Instead,
  Seq 2  : Robots will soon be able to do so, with the addition of these tools to the development of robot tools.






















  Seq 3  : Robots will soon be able to access their data in-depth and user-friendly ways. For instance, it could be used to easily access a user›s location, for example, in a Google Maps dashboard.



Test 5 — Temperature Effect
Compares temperature=0.3 (low) vs temperature=1.5 (high). Temperature scales the probability distribution over the next token:

Low temp → model plays it safe, picks high-probability (repetitive) words
High temp → probabilities are flattened, model takes wilder guesses

In [11]:

# ════════════════════════════════════════════════════════════
#  TEST 5 · Temperature — focused (0.3) vs creative (1.5)
# ════════════════════════════════════════════════════════════
prompt5 = "Deep learning models can"
print("─" * 62)
print("  TEST 5 │ Temperature Effect (do_sample=True)")
print("  ℹ️  Low temp = repetitive/safe. High temp = wild/creative.")
print("─" * 62)
print(f"  Prompt : \"{prompt5}\"")

r_low  = generator(prompt5, max_length=40, num_return_sequences=1,
                   do_sample=True, temperature=0.3)
r_high = generator(prompt5, max_length=40, num_return_sequences=1,
                   do_sample=True, temperature=1.5)


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


──────────────────────────────────────────────────────────────
  TEST 5 │ Temperature Effect (do_sample=True)
  ℹ️  Low temp = repetitive/safe. High temp = wild/creative.
──────────────────────────────────────────────────────────────
  Prompt : "Deep learning models can"


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [12]:

print(f"  temp=0.3: {r_low[0]['generated_text']}")


  temp=0.3: Deep learning models can be used to create a new learning model.





























In [13]:
print(f"  temp=1.5: {r_high[0]['generated_text']}")

  temp=1.5: Deep learning models can tell, say, how people act with emotion as long as their emotion plays an integrated role. To test what․ they use this type of emotion for emotion, one simple task


Test 6 — Greedy vs Sampling

do_sample=False → Greedy decoding: always picks the single most likely next token. Fully deterministic — same output every run.
do_sample=True → Random sampling: picks from the probability distribution, so output varies each run.

In [14]:
# ════════════════════════════════════════════════════════════
#  TEST 6 · Greedy vs Sampling (do_sample flag)
# ════════════════════════════════════════════════════════════
prompt6 = "The best way to learn programming is"
print("─" * 62)
print("  TEST 6 │ Greedy Decoding vs Random Sampling")
print("  ℹ️  Greedy always picks the highest-prob token → deterministic")
print("─" * 62)
print(f"  Prompt : \"{prompt6}\"")

r_greedy  = generator(prompt6, max_length=40, do_sample=False)   # greedy
r_sampled = generator(prompt6, max_length=40, do_sample=True, temperature=1.0)

print(f"  greedy  : {r_greedy[0]['generated_text']}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


──────────────────────────────────────────────────────────────
  TEST 6 │ Greedy Decoding vs Random Sampling
  ℹ️  Greedy always picks the highest-prob token → deterministic
──────────────────────────────────────────────────────────────
  Prompt : "The best way to learn programming is"


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


  greedy  : The best way to learn programming is to learn from the experience of the programmer.


























In [15]:
print(f"  sampled : {r_sampled[0]['generated_text']}")

  sampled : The best way to learn programming is using the best techniques, with all sorts of programming languages, so that you're already familiar with how your students can learn. With C#, you can learn programming


Test 7 — Tokenisation Deep-Dive
Uses the raw AutoTokenizer to show how GPT-2 splits text into subword tokens using Byte-Pair Encoding (BPE). Key insight: word count ≠ token count. For example, "Unbelievably" might be split into ["Un", "believ", "ably"] — three tokens from one word.

In [16]:
# ════════════════════════════════════════════════════════════
#  TEST 7 · Tokenisation deep-dive
# ════════════════════════════════════════════════════════════
print("─" * 62)
print("  TEST 7 │ Tokenisation Peek")
print("  ℹ️  Words ≠ tokens. GPT-2 uses Byte-Pair Encoding (BPE).")
print("─" * 62)

tokenizer = AutoTokenizer.from_pretrained("distilgpt2")

sentences = [
    "AI is transforming industries by automating tasks",
    "Unbelievably, transformers revolutionised NLP overnight!",
]

for sentence in sentences:
    token_ids = tokenizer.encode(sentence)
    tokens    = [tokenizer.decode([t]) for t in token_ids]
    print(f"\n  Sentence  : {sentence}")
    print(f"  Token IDs : {token_ids}")
    print(f"  Tokens    : {tokens}")
    print(f"  Stats     : {len(sentence.split())} words → {len(token_ids)} tokens")

print()

──────────────────────────────────────────────────────────────
  TEST 7 │ Tokenisation Peek
  ℹ️  Words ≠ tokens. GPT-2 uses Byte-Pair Encoding (BPE).
──────────────────────────────────────────────────────────────

  Sentence  : AI is transforming industries by automating tasks
  Token IDs : [20185, 318, 25449, 11798, 416, 3557, 803, 8861]
  Tokens    : ['AI', ' is', ' transforming', ' industries', ' by', ' autom', 'ating', ' tasks']
  Stats     : 7 words → 8 tokens

  Sentence  : Unbelievably, transformers revolutionised NLP overnight!
  Token IDs : [3118, 6667, 11203, 1346, 11, 6121, 364, 5854, 1417, 399, 19930, 13417, 0]
  Tokens    : ['Un', 'bel', 'iev', 'ably', ',', ' transform', 'ers', ' revolution', 'ised', ' N', 'LP', ' overnight', '!']
  Stats     : 5 words → 13 tokens



Summary Table at the End
Prints a quick-reference cheat sheet mapping each parameter to its observed effect, making it easy to remember the key takeaways from all 7 tests.

In [17]:
# ════════════════════════════════════════════════════════════
#  SUMMARY TABLE
# ════════════════════════════════════════════════════════════
print("=" * 62)
print("  WHAT TO OBSERVE — Key Takeaways")
print("=" * 62)
summary = [
    ("max_length ↑",         "Longer output; model may drift off-topic"),
    ("num_return_sequences↑","Multiple diverse completions in one call"),
    ("temperature ↓ (0.1)",  "Safe, repetitive, predictable text"),
    ("temperature ↑ (1.5)",  "Creative, sometimes incoherent text"),
    ("do_sample=False",      "Greedy — same output every run"),
    ("do_sample=True",       "Stochastic — different each time"),
    ("Tokenisation",         "Subword tokens, not whole words (BPE)"),
]
print(f"  {'Parameter':<28} {'Effect'}")
print("  " + "-" * 56)
for param, effect in summary:
    print(f"  {param:<28} {effect}")
print("=" * 62)
print("  ✅  All tests complete!")
print("=" * 62)

  WHAT TO OBSERVE — Key Takeaways
  Parameter                    Effect
  --------------------------------------------------------
  max_length ↑                 Longer output; model may drift off-topic
  num_return_sequences↑        Multiple diverse completions in one call
  temperature ↓ (0.1)          Safe, repetitive, predictable text
  temperature ↑ (1.5)          Creative, sometimes incoherent text
  do_sample=False              Greedy — same output every run
  do_sample=True               Stochastic — different each time
  Tokenisation                 Subword tokens, not whole words (BPE)
  ✅  All tests complete!


 This code is a parameter sensitivity study — it isolates one variable at a time so you can clearly see how each knob changes the model's behavior.

<h2>Section B: Prompt Engineering</h2>

1. The word_count() Helper
A simple utility that splits the generated text by spaces and counts the resulting list. Used to check whether the summarisation output actually stays within the ≤ 30 word target.

In [19]:
# ────────────────────────────────────────────────────────────
#  HELPER — word counter to check output length
# ────────────────────────────────────────────────────────────
def word_count(text: str) -> int:
    return len(text.split())

2. The show_b() Helper
    This is the Section B version of the original show() function. The key difference is it accepts lists of prompts and results and uses zip() to pair each prompt with its corresponding output, printing them together for easy side-by-side comparison. The word count is shown alongside each output

In [20]:
def show_b(title: str, prompts: list, results: list, notes: str = ""):
    """Display side-by-side prompt comparisons for Section B."""
    print("─" * 62)
    print(f"  {title}")
    if notes:
        print(f"  ℹ️  {notes}")
    print("─" * 62)
    for i, (prompt, result) in enumerate(zip(prompts, results), 1):
        output = result[0]['generated_text']
        wc     = word_count(output)
        print(f"\n  Prompt v{i} : \"{prompt}\"")
        print(f"  Output    : {output}")
        print(f"  Word count: {wc} words")
    print()

Task 1 — Summarisation
Three prompts are designed with increasing levels of instruction clarity:

        v1 - NothingBaseline           - see what the model does freely
        v2 - In one sentence           - Gives a format constraint
        v3 - Source text + word limit  - Anchors the model to specific content


All three use do_sample=False (greedy decoding) so any output differences come purely from the prompt wording, not randomness.

In [30]:

# ════════════════════════════════════════════════════════════
#  TASK 1 · SUMMARISATION
#  Goal : Keep output ≤ 30 words.
#  Strategy : Compare vague vs specific instruction phrasing.
# ════════════════════════════════════════════════════════════
print("━" * 62)
print("  TASK 1 │ SUMMARISATION  (target: ≤ 30 words output)")
print("━" * 62)

# Prompt v1 — Vague: no length or style instruction
sum_prompt_v1 = "Summarize AI"

# Prompt v2 — Clear: specifies brevity and a sentence cap
sum_prompt_v2 = "In one sentence, briefly summarize Vector embedding"

# Prompt v3 — Structured: gives context + explicit instruction
sum_prompt_v3 = (
    "Food waste carries a significant environmental toll by squandering resources like water, fertilizer, and fuel used in production, and if treated as a country, it would be the world's third-largest emitter of greenhouse gases. "
    "Summarize this in 20 words or fewer:"
)

sum_prompts = [sum_prompt_v1, sum_prompt_v2, sum_prompt_v3]
sum_results = [
    generator(p, max_length=30, num_return_sequences=1, do_sample=False)
    for p in sum_prompts
]

show_b(
    title   = "SUMMARISATION — Prompt Comparison",
    prompts = sum_prompts,
    results = sum_results,
    notes   = "v1=vague | v2=adds brevity cue | v3=context + length constraint"
)

print("Observation:")


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  TASK 1 │ SUMMARISATION  (target: ≤ 30 words output)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


──────────────────────────────────────────────────────────────
  SUMMARISATION — Prompt Comparison
  ℹ️  v1=vague | v2=adds brevity cue | v3=context + length constraint
──────────────────────────────────────────────────────────────

  Prompt v1 : "Summarize AI"
  Output    : Summarize AI.

























  Word count: 2 words

  Prompt v2 : "In one sentence, briefly summarize Vector embedding"
  Output    : In one sentence, briefly summarize Vector embedding.



The first sentence is a simple example of the concept of embedding.
The
  Word count: 20 words

  Prompt v3 : "Food waste carries a significant environmental toll by squandering resources like water, fertilizer, and fuel used in production, and if treated as a country, it would be the world's third-largest emitter of greenhouse gases. Summarize this in 20 words or fewer:"
  Output    : Food waste carries a significant environmental toll by squandering resources like water, fertilizer, and fuel used in production, and i

Task 2 — Q&A
Three different prompt strategies for factual answering:

v1 — A plain question. DistilGPT-2 often just rephrases or echoes the question back because it wasn't trained as a Q&A model.
v2 — Adds a role ("As an AI expert") and a format instruction ("one sentence"). This nudges the model toward a more informative, confident tone.
v3 — A sentence-completion style. Instead of asking a question, you give the model the beginning of the answer. This is the most effective trick for small models — it removes ambiguity about what kind of response is expected.

In [32]:

# ════════════════════════════════════════════════════════════
#  TASK 2 · QUESTION & ANSWER (Q&A)
#  Goal : Elicit a factual, direct answer.
#  Strategy : Compare open-ended vs answer-framing prompts.
# ════════════════════════════════════════════════════════════
print("━" * 62)
print("  TASK 2 │ Q&A  (factual question answering)")
print("━" * 62)

# Prompt v1 — Plain question only
qa_prompt_v1 = "What is Mathematics?"

# Prompt v2 — Role + question format
qa_prompt_v2 = "As an expert mathematian, answer in one sentence: What is Maths?"

# Prompt v3 — Fill-in-the-blank style (strong answer framing)
qa_prompt_v3 = "Calculas is a branch of Maths that"

qa_prompts = [qa_prompt_v1, qa_prompt_v2, qa_prompt_v3]
qa_results = [
    generator(p, max_length=55, num_return_sequences=1, do_sample=False)
    for p in qa_prompts
]

show_b(
    title   = "Q&A — Prompt Comparison",
    prompts = qa_prompts,
    results = qa_results,
    notes   = "v1=plain | v2=role+format | v3=sentence-completion framing"
)

print("  📝 Observation:")
print("     v1 (plain)       → Model may repeat or rephrase the question.")
print("     v2 (role+format) → 'Expert' framing encourages factual language.")
print("     v3 (completion)  → Sentence-start forces the model straight to answer.")
print()

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  TASK 2 │ Q&A  (factual question answering)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


──────────────────────────────────────────────────────────────
  Q&A — Prompt Comparison
  ℹ️  v1=plain | v2=role+format | v3=sentence-completion framing
──────────────────────────────────────────────────────────────

  Prompt v1 : "What is Mathematics?"
  Output    : What is Mathematics?



















































  Word count: 3 words

  Prompt v2 : "As an expert mathematian, answer in one sentence: What is Maths?"
  Output    : As an expert mathematian, answer in one sentence: What is Maths?



The answer is that the answer is that the answer is that the answer is that the answer is that the answer is that the answer is that the answer is that the answer is
  Word count: 46 words

  Prompt v3 : "Calculas is a branch of Maths that"
  Output    : Calculas is a branch of Maths that is used to calculate the number of times a given number of times a given number of times a given number of times a given number of times a given number of times a given number of time

5. Task 3 — Creative Text (Poem)
Each version adds more creative scaffolding:

v1 — Bare request. Small models like DistilGPT-2 tend to produce prose rather than actual verse because the instruction is too open-ended.
v2 — Specifies "4-line" and "rhyming". These structural keywords push the model toward verse-shaped output.
v3 — Provides the first line of the poem. This is the strongest technique — the model simply continues from where you left off, inheriting the rhythm, theme, and style you set.
Higher temperature is intentional here — creativity tasks benefit from some randomness rather than always picking the safest next word.

In [35]:

# ════════════════════════════════════════════════════════════
#  TASK 3 · CREATIVE TEXT GENERATION
#  Goal : Generate a 4-line poem on AI.
#  Strategy : Compare bare request vs structured creative prompts.
# ════════════════════════════════════════════════════════════
print("━" * 62)
print("  TASK 3 │ CREATIVE TEXT  (4-line poem on AI)")
print("━" * 62)

# Prompt v1 — Bare request
creative_prompt_v1 = "Write a poem about artificial intelligence."

# Prompt v2 — Specifies structure (4 lines) + mood
creative_prompt_v2 = "Write a 4-line rhyming poem about artificial intelligence and the future:"

# Prompt v3 — Gives the first line as a creative anchor
creative_prompt_v3 = (
    "Complete this 4-line poem about AI:\n"
    "In circuits and code, a new mind is born,\n"
)

creative_prompts = [creative_prompt_v1, creative_prompt_v2, creative_prompt_v3]
creative_results = [
    generator(
        p,
        max_length      = 80,
        num_return_sequences = 1,
        do_sample       = True,
        temperature     = 0.9      # higher temp for more creative output
    )
    for p in creative_prompts
]

show_b(
    title   = "CREATIVE TEXT — Prompt Comparison",
    prompts = creative_prompts,
    results = creative_results,
    notes   = "v1=bare | v2=structure+mood | v3=anchor line provided"
)

print("  📝 Observation:")
print("     v1 (bare)    → Output may be prose-like, not poem-shaped.")
print("     v2 (specific)→ 'Rhyming' cue pushes toward verse structure.")
print("     v3 (anchor)  → Giving line 1 strongly constrains style & theme.")
print()


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  TASK 3 │ CREATIVE TEXT  (4-line poem on AI)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


──────────────────────────────────────────────────────────────
  CREATIVE TEXT — Prompt Comparison
  ℹ️  v1=bare | v2=structure+mood | v3=anchor line provided
──────────────────────────────────────────────────────────────

  Prompt v1 : "Write a poem about artificial intelligence."
  Output    : Write a poem about artificial intelligence.



The goal of this project was to document the most powerful tools for analyzing the human brain through human brain-computer interaction (BDI). The work was described in the Proceedings of the National Academy of Sciences.
The goal of this project was to document the most powerful tools for analyzing the human brain through human brain-computer interaction (BDI).
  Word count: 63 words

  Prompt v2 : "Write a 4-line rhyming poem about artificial intelligence and the future:"
  Output    : Write a 4-line rhyming poem about artificial intelligence and the future: "I am a big believer in AI."
  Word count: 18 words

  Prompt v3 : "Complete this 4-line 

6. The Comparison Loop Pattern
All three tasks follow the same pattern:
A list comprehension runs the generator on every prompt and collects all results into one list. This keeps the code clean and avoids repeating the generator() call three separate times.

In [36]:

# ════════════════════════════════════════════════════════════
#  PROMPT COMPARISON SUMMARY TABLE
# ════════════════════════════════════════════════════════════
print("=" * 62)
print("  SECTION B — Prompt Engineering Summary")
print("=" * 62)
pe_summary = [
    ("Task",          "Prompt Style",      "Key Effect"),
    ("─" * 14,        "─" * 18,            "─" * 24),
    ("Summarisation", "Vague (v1)",         "Verbose, unfocused output"),
    ("",              "Clear (v2)",         "Shorter, more on-topic"),
    ("",              "Context+limit (v3)", "Best anchored summary"),
    ("Q&A",           "Plain (v1)",         "May echo the question"),
    ("",              "Role+format (v2)",   "More factual tone"),
    ("",              "Completion (v3)",    "Direct, concise answer"),
    ("Creative",      "Bare (v1)",          "Prose drift, no structure"),
    ("",              "Structured (v2)",    "More verse-like output"),
    ("",              "Anchor line (v3)",   "Strongest style control"),
]
for row in pe_summary:
    print(f"  {row[0]:<14} {row[1]:<20} {row[2]}")
print()

  SECTION B — Prompt Engineering Summary
  Task           Prompt Style         Key Effect
  ────────────── ──────────────────   ────────────────────────
  Summarisation  Vague (v1)           Verbose, unfocused output
                 Clear (v2)           Shorter, more on-topic
                 Context+limit (v3)   Best anchored summary
  Q&A            Plain (v1)           May echo the question
                 Role+format (v2)     More factual tone
                 Completion (v3)      Direct, concise answer
  Creative       Bare (v1)            Prose drift, no structure
                 Structured (v2)      More verse-like output
                 Anchor line (v3)     Strongest style control



<h2> Section C: Embeddings with Gensim</h2>

Setup & Model Loading
StepDetailLibrarygensim.downloader fetches pre-trained GloVe weights automaticallyModelglove-wiki-gigaword-50 — trained on Wikipedia + Gigaword, 50-dimensional vectorsSize~400K vocabulary words


In [1]:
# ============================================================
#   Embeddings with Gensim
#   - Word Embeddings (GloVe: glove-wiki-gigaword-50)
#   - Sentence-Level Embeddings (Average Word Vectors)
# ============================================================

import gensim.downloader as api
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings("ignore")


In [2]:
# ─────────────────────────────────────────────────────────────
# 1. LOAD PRE-TRAINED GloVe MODEL
# ─────────────────────────────────────────────────────────────

print("Loading GloVe embeddings (glove-wiki-gigaword-50)...")
model = api.load("glove-wiki-gigaword-50")
print(f"Model loaded! Vocabulary size: {len(model)}, Vector size: {model.vector_size}\n")



Loading GloVe embeddings (glove-wiki-gigaword-50)...
[==================================================] 100.0% 66.0/66.0MB downloaded
Model loaded! Vocabulary size: 400000, Vector size: 50



<h4>Word Embeddings</h4>

model[word] — fetches the raw 50-dim NumPy vector for a word
model.most_similar(word, topn=5) — uses cosine similarity internally to find the top-N nearest neighbors in vector space

In [3]:
# ─────────────────────────────────────────────────────────────
# 2. WORD EMBEDDINGS
# ─────────────────────────────────────────────────────────────

selected_words = ["king", "queen", "diamond"]

print("=" * 60)
print("SECTION 1: WORD EMBEDDINGS")
print("=" * 60)

for word in selected_words:

    print(f"Word: '{word}'")
    print(f"{'─' * 40}")

    # ── First 10 values of the word vector
    vector = model[word]
    print(f"\n  First 10 values of vector:")
    print(f"  {np.round(vector[:10], 4)}")

    # ── Top 5 most similar words
    similar_words = model.most_similar(word, topn=5)
    print(f"\n  Top 5 most similar words:")
    print(f"  {'Word':<20} {'Similarity Score':>16}")
    print(f"  {'----':<20} {'----------------':>16}")
    for sim_word, score in similar_words:
        print(f"  {sim_word:<20} {score:>16.4f}")

print("\n")


SECTION 1: WORD EMBEDDINGS
Word: 'king'
────────────────────────────────────────

  First 10 values of vector:
  [ 0.5045  0.6861 -0.5952 -0.0228  0.6005 -0.135  -0.0881  0.4738 -0.618
 -0.3101]

  Top 5 most similar words:
  Word                 Similarity Score
  ----                 ----------------
  prince                         0.8236
  queen                          0.7839
  ii                             0.7746
  emperor                        0.7736
  son                            0.7667
Word: 'queen'
────────────────────────────────────────

  First 10 values of vector:
  [ 0.3785  1.8233 -1.2648 -0.1043  0.3583  0.6003 -0.1754  0.8377 -0.0568
 -0.758 ]

  Top 5 most similar words:
  Word                 Similarity Score
  ----                 ----------------
  princess                       0.8515
  lady                           0.8051
  elizabeth                      0.7873
  king                           0.7839
  prince                         0.7822
Word: 'diamond'
─

<h4>Sentence Embeddings</h4>
get_sentence_vector() function logic:
Tokenize → Filter OOV words → Average valid vectors → Return mean vector

Average pooling is a simple but effective baseline — no model fine-tuning needed
Words not in vocabulary (OOV) are silently skipped
If all words are OOV, a zero vector is returned as a safe fallback

In [9]:


# ─────────────────────────────────────────────────────────────
# 3. SENTENCE-LEVEL EMBEDDINGS
# ─────────────────────────────────────────────────────────────

sentences = [
    "Artificial intelligence is transforming the world",
    "Machine learning models learn patterns from data",
    "Diamonds and rubies are precious stones",
    "Gold and silver are precious metals used in jewellery",
    "Deep learning enables computers to understand images",
]

print("=" * 60)
print("SECTION 2: SENTENCE-LEVEL EMBEDDINGS")
print("=" * 60)


def get_sentence_vector(sentence: str, glove_model) -> np.ndarray:
    """
    Compute a sentence vector by averaging the GloVe vectors
    of all known words in the sentence.

    Parameters
    ----------
    sentence    : Raw sentence string.
    glove_model : Loaded Gensim KeyedVectors model.

    Returns
    -------
    np.ndarray  : Mean word vector (zeros if no known words found).
    """
    tokens = sentence.lower().split()
    # Keep only tokens that exist in the vocabulary
    valid_vectors = [glove_model[token] for token in tokens if token in glove_model]

    if valid_vectors:
        return np.mean(valid_vectors, axis=0)          # Average pooling
    else:
        return np.zeros(glove_model.vector_size)       # Fallback: zero vector


# ── Compute sentence vectors
print("\nSentences used:")
sentence_vectors = []
for i, sentence in enumerate(sentences, 1):
    vec = get_sentence_vector(sentence, model)
    sentence_vectors.append(vec)
    print(f"  S{i}: {sentence}")

sentence_matrix = np.array(sentence_vectors)          # Shape: (5, 50)



SECTION 2: SENTENCE-LEVEL EMBEDDINGS

Sentences used:
  S1: Artificial intelligence is transforming the world
  S2: Machine learning models learn patterns from data
  S3: Diamonds and rubies are precious stones
  S4: Gold and silver are precious metals used in jewellery
  S5: Deep learning enables computers to understand images


Similarity Matrix:

cosine_similarity(matrix) from sklearn computes an N×N matrix
The diagonal is always 1.0 (a sentence compared to itself)
Sentences about AI/ML (S1, S2, S5) should cluster together, while jewellery sentences (S3, S4) will form a separate cluster


In [10]:

# ── Cosine Similarity Matrix
similarity_matrix = cosine_similarity(sentence_matrix)

print(f"\n{'─' * 60}")
print("Cosine Similarity Matrix (Sentences vs Sentences):")
print(f"{'─' * 60}\n")

# Header row
header = f"{'':6}" + "".join(f"  S{i:<6}" for i in range(1, len(sentences) + 1))
print(header)
print("  " + "─" * (len(header) - 2))

# Matrix rows
for i, row in enumerate(similarity_matrix):
    row_str = f"  S{i+1:<4}" + "".join(f"  {score:.4f}" for score in row)
    print(row_str)

print(f"\n{'─' * 60}")
print("Interpretation Guide:")
print("  Score ~1.00  → Very high similarity")
print("  Score ~0.75  → Moderate similarity")
print("  Score ~0.50  → Low similarity")
print("  Score ~0.00  → No similarity")
print(f"{'─' * 60}\n")


# ── Most and Least Similar Sentence Pairs
print("Top 3 Most Similar Sentence Pairs:")
pairs = []
for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        pairs.append((similarity_matrix[i][j], i + 1, j + 1))

pairs.sort(reverse=True)

for score, i, j in pairs[:3]:
    print(f"  S{i} ↔ S{j}  |  Score: {score:.4f}")
    print(f"    S{i}: {sentences[i-1]}")
    print(f"    S{j}: {sentences[j-1]}\n")

print("Top 3 Least Similar Sentence Pairs:")
for score, i, j in pairs[-3:]:
    print(f"  S{i} ↔ S{j}  |  Score: {score:.4f}")
    print(f"    S{i}: {sentences[i-1]}")
    print(f"    S{j}: {sentences[j-1]}\n")


────────────────────────────────────────────────────────────
Cosine Similarity Matrix (Sentences vs Sentences):
────────────────────────────────────────────────────────────

        S1       S2       S3       S4       S5     
  ─────────────────────────────────────────────────
  S1     1.0000  0.8065  0.5692  0.7349  0.8348
  S2     0.8065  1.0000  0.5178  0.6911  0.9291
  S3     0.5692  0.5178  1.0000  0.8678  0.5472
  S4     0.7349  0.6911  0.8678  1.0000  0.6493
  S5     0.8348  0.9291  0.5472  0.6493  1.0000

────────────────────────────────────────────────────────────
Interpretation Guide:
  Score ~1.00  → Very high similarity
  Score ~0.75  → Moderate similarity
  Score ~0.50  → Low similarity
  Score ~0.00  → No similarity
────────────────────────────────────────────────────────────

Top 3 Most Similar Sentence Pairs:
  S2 ↔ S5  |  Score: 0.9291
    S2: Machine learning models learn patterns from data
    S5: Deep learning enables computers to understand images

  S3 ↔ S4  |  S

<h2>Section D: Application Exploration</h2>

Use tasks from the Hugging Face pipeline (translation, summarisation, or sentiment classification) to improve productivity in SLDC.
    1) Identify the tasks that can be used at different SLDC phases
    2) Identify the hugging face models that can be used for the tasks
    3) Write sample exmaple code to explain the different SLDC phases using these models and hugging face tasks



 
| Phase          | Hugging Face Task          | Description                                                                 |
|----------------|----------------------------|-----------------------------------------------------------------------------|
| Planning       | Sentiment classification   | Analyse stakeholder surveys & feedback to surface pain points early         |
| Requirements   | Summarisation              | Condense lengthy requirement docs & user stories into concise specs         |
|                | Translation                | Localise requirement documents for distributed, multilingual teams          |
| Design         | Summarisation              | Auto-generate executive summaries from architecture & design documents      |
| Implementation | Translation                | Translate inline comments & docs so global dev teams share one codebase     |
| Testing        | Sentiment classification   | Triage bug reports & QA feedback — flag critical issues automatically       |
| Deployment     | Summarisation              | Auto-generate release notes & changelogs from commit histories              |
| Maintenance    | Sentiment classification   | Monitor live reviews & support tickets — detect satisfaction trends         |
|                | Translation                | Localise error messages & UI strings for post-deployment support      

| SDLC Phase     | Task             | Model Used                        | Productivity Gain                                                                 |
|----------------|------------------|-----------------------------------|-----------------------------------------------------------------------------------|
| Planning       | Sentiment        | `distilbert-base-uncased-finetuned-sst-2-english` | Surfaces critical pain points from hundreds of survey responses in seconds        |
| Requirements   | Summarisation    | `facebook/bart-large-cnn`         | Turns verbose user stories into concise sprint-ready specs                        |
|                | Translation      | `Helsinki-NLP/opus-mt-en-*`       | Distributes requirement docs to global teams without manual translation           |
| Design         | Summarisation    | `facebook/bart-large-cnn`         | Auto-writes executive design summaries for non-technical stakeholders             |
| Implementation | Translation      | `Helsinki-NLP/opus-mt-en-*`       | Localises docstrings and comments for multilingual dev teams                      |
| Testing        | Sentiment        | `distilbert-base-uncased-finetuned-sst-2-english` | Auto-triages bug reports by severity — critical issues get escalated instantly    |
| Deployment     | Summarisation    | `facebook/bart-large-cnn`         | Generates human-readable release notes from raw commit logs                       |
| Maintenance    | Sentiment        | `distilbert-base-uncased-finetuned-sst-2-english` | Monitors live user reviews to detect satisfaction drops before they escalate      |
|                | Translation      | `Helsinki-NLP/opus-mt-en-*`       | Localises UI error messages and support responses post-deployment                 |

<h3>Phase 1 — Planning: Stakeholder survey analysis</h3>

In [12]:
from transformers import pipeline

analyser = pipeline("sentiment-analysis",
                    model="distilbert-base-uncased-finetuned-sst-2-english")

# Stakeholder responses collected during the planning workshop
stakeholder_feedback = [
    "The current checkout flow is painfully slow and confusing.",
    "I love how the dashboard gives me instant visibility.",
    "Reporting takes forever and the exports never work correctly.",
    "The mobile experience is much better than last quarter.",
    "We desperately need offline mode — the app is useless without internet.",
]

print("=== Planning Phase: Stakeholder Sentiment Analysis ===\n")
results = analyser(stakeholder_feedback)

for text, result in zip(stakeholder_feedback, results):
    tag   = result["label"]
    score = result["score"]
    flag  = "⚠ Prioritise" if tag == "NEGATIVE" and score > 0.90 else ""
    print(f"[{tag:<8} | {score:.2f}] {flag}")
    print(f"  → {text}\n")

2026-05-05 07:05:35.182304: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-05 07:05:35.762466: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777964735.928311      75 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777964735.978449      75 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777964736.380056      75 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

=== Planning Phase: Stakeholder Sentiment Analysis ===

[NEGATIVE | 1.00] ⚠ Prioritise
  → The current checkout flow is painfully slow and confusing.

[POSITIVE | 1.00] 
  → I love how the dashboard gives me instant visibility.

[NEGATIVE | 1.00] ⚠ Prioritise
  → Reporting takes forever and the exports never work correctly.

[POSITIVE | 1.00] 
  → The mobile experience is much better than last quarter.

[NEGATIVE | 1.00] ⚠ Prioritise
  → We desperately need offline mode — the app is useless without internet.



<h3>Phase 2 — Requirements: Condensing user stories</h3>

In [16]:
from transformers import pipeline

summariser = pipeline("summarization",
                      model="facebook/bart-large-cnn",
                      min_length=30,
                      max_length=80)

user_story = """
    As a returning customer, I want to view my full order history with
    filtering by date range, product category, and order status (delivered,
    pending, cancelled), so that I can quickly locate past purchases for
    returns or reorders. The list should be paginated (20 items per page),
    support export to CSV, and display estimated delivery dates alongside
    actual delivery dates where applicable. Search by order ID or product
    name should also be available in the header.
"""



In [17]:
print("=== Requirements Phase: User Story Summarisation ===\n")
print("Original user story:")
print(user_story.strip())

summary = summariser(user_story)[0]["summary_text"]
print(f"\nCondensed spec (for sprint board):\n  → {summary}")

=== Requirements Phase: User Story Summarisation ===

Original user story:
As a returning customer, I want to view my full order history with
    filtering by date range, product category, and order status (delivered,
    pending, cancelled), so that I can quickly locate past purchases for
    returns or reorders. The list should be paginated (20 items per page),
    support export to CSV, and display estimated delivery dates alongside
    actual delivery dates where applicable. Search by order ID or product
    name should also be available in the header.

Condensed spec (for sprint board):
  → The list should be paginated (20 items per page), support export to CSV, and display estimated delivery dates alongside actual delivery dates. Search by order ID or product name should also be available in the header.


<h2>Phase 3 - Design - Summarisation  - Auto-generate executive summaries from architecture & design documents</h2>

In [1]:
# ============================================================
#   SDLC — Design Phase
#   Auto-Generate Executive Design Summaries
#   Model : facebook/bart-large-cnn
#   Task  : Summarisation → Non-technical stakeholder briefs
# ============================================================

from transformers import pipeline
from datetime import datetime

/usr/local/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
2026-05-06 06:46:20.623658: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-06 06:46:20.636275: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778049980.654378      75 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778049980.660254      75 cuda_blas.cc:1407] Unable to reg

In [2]:
# ─────────────────────────────────────────────────────────────
# 1. LOAD MODEL
# ─────────────────────────────────────────────────────────────

print("Loading summarisation model (facebook/bart-large-cnn)...")
summariser = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
)
print("Model ready.\n")

Loading summarisation model (facebook/bart-large-cnn)...


/usr/local/lib/python3.10/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model ready.



In [3]:
# ─────────────────────────────────────────────────────────────
# 2. SAMPLE TECHNICAL DESIGN DOCUMENTS
#    (As written by engineers / architects)
# ─────────────────────────────────────────────────────────────

design_documents = [
    {
        "title"    : "Authentication Service — Architecture Design",
        "audience" : "CTO / Product Owner",
        "content"  : """
            The authentication service will be implemented as a stateless
            microservice using OAuth 2.0 with PKCE flow for all public-facing
            clients. JWTs will be signed with RS256 asymmetric keys rotated
            every 24 hours via AWS KMS. Token expiry is set to 15 minutes for
            access tokens and 7 days for refresh tokens stored in HttpOnly
            cookies. The service will expose four REST endpoints: /token,
            /refresh, /revoke, and /introspect, all behind an NGINX API
            gateway with rate limiting of 100 requests per minute per IP.
            Redis will be used for token blacklisting with a TTL matching the
            original token expiry. Horizontal scaling is supported via
            stateless design. Expected p99 latency under load is under 120ms.
            A circuit breaker pattern via Resilience4j will prevent cascading
            failures if the identity provider becomes unavailable.
        """
    },
    {
        "title"    : "Database Schema — Order Management System",
        "audience" : "Project Manager / Business Analyst",
        "content"  : """
            The order management database will use PostgreSQL 15 with a
            normalised schema across six core tables: customers, addresses,
            products, orders, order_items, and payments. The orders table
            will carry a status ENUM with seven states: draft, confirmed,
            processing, shipped, delivered, cancelled, and refunded. Foreign
            key constraints enforce referential integrity between orders and
            customers. Indexes are applied to order status, created_at, and
            customer_id columns to support the most frequent query patterns.
            Soft deletes are implemented via a deleted_at timestamp column
            rather than physical row deletion to preserve audit history.
            Partitioning by created_at will be applied to the order_items
            table once row count exceeds five million records. Connection
            pooling is handled by PgBouncer with a max pool size of 100
            per application node. Read replicas will serve reporting queries
            to avoid contention with transactional workloads.
        """
    },
    {
        "title"    : "Frontend Architecture — Customer Portal Redesign",
        "audience" : "Marketing Director / UX Sponsor",
        "content"  : """
            The customer portal will be rebuilt as a React 18 single-page
            application using Next.js 14 with the App Router for server-side
            rendering of SEO-critical pages. State management will use Zustand
            for lightweight global state alongside React Query for server state
            and cache invalidation. Component design follows atomic design
            principles with a Storybook library of 40+ reusable components
            built on top of Radix UI primitives for accessibility compliance
            with WCAG 2.1 AA. Code splitting via dynamic imports reduces the
            initial JavaScript bundle to under 120 KB gzipped. The design
            system tokens are sourced from a Figma variables file and exported
            as CSS custom properties at build time, ensuring pixel-perfect
            parity between design and implementation. Lighthouse performance
            score target is 90+ on mobile. Deployment is via Vercel with
            preview environments generated per pull request for stakeholder
            review before merging.
        """
    },
    {
        "title"    : "API Design — Payment Integration Gateway",
        "audience" : "Finance Director / Compliance Officer",
        "content"  : """
            The payment gateway integration will use Stripe as the primary
            processor with PayPal as a failover option. Payment intents will
            be created server-side to prevent client-side manipulation of
            amounts. All card data is handled exclusively by Stripe Elements,
            ensuring the application never touches raw PAN data and maintaining
            PCI DSS SAQ A compliance. Webhooks from Stripe will be verified
            using HMAC signatures and processed via an idempotent event handler
            to prevent duplicate charges on retry. Refunds, partial captures,
            and dispute handling are exposed through an internal admin API
            restricted to staff roles. All payment events are logged to an
            append-only audit table with user ID, timestamp, amount, currency,
            and outcome. Currency conversion is handled at the Stripe level
            using dynamic currency conversion. Failed payment retry logic
            follows an exponential backoff strategy with a maximum of three
            retries over 72 hours before the order is cancelled automatically.
        """
    },
]

In [4]:

# ─────────────────────────────────────────────────────────────
# 3. SUMMARISATION CONFIGS PER AUDIENCE TYPE
#    Different stakeholders need different summary lengths
# ─────────────────────────────────────────────────────────────

AUDIENCE_CONFIG = {
    # C-Suite → very brief, strategic framing only
    "CTO / Product Owner"           : {"min_length": 40, "max_length": 80 },
    # PM / BA → moderate detail, enough to brief their team
    "Project Manager / Business Analyst" : {"min_length": 55, "max_length": 110},
    # Marketing / UX sponsors → outcome-focused, no jargon
    "Marketing Director / UX Sponsor"    : {"min_length": 50, "max_length": 95 },
    # Finance / Compliance → risk and regulation focus
    "Finance Director / Compliance Officer": {"min_length": 60, "max_length": 120},
}


In [5]:
# ─────────────────────────────────────────────────────────────
# 4. EXECUTIVE SUMMARY GENERATOR
# ─────────────────────────────────────────────────────────────

def generate_executive_summary(
    doc: dict,
    summariser_pipeline,
    audience_config: dict,
) -> dict:
    """
    Generate a non-technical executive summary from a design document.

    Parameters
    ----------
    doc               : dict with keys 'title', 'audience', 'content'
    summariser_pipeline: loaded Hugging Face summarisation pipeline
    audience_config   : dict mapping audience label to min/max_length

    Returns
    -------
    dict with original metadata plus generated 'summary' and 'generated_at'
    """
    config = audience_config.get(
        doc["audience"],
        {"min_length": 50, "max_length": 100}        # sensible default
    )

    raw_summary = summariser_pipeline(
        doc["content"].strip(),
        min_length=config["min_length"],
        max_length=config["max_length"],
        do_sample=False,                              # deterministic output
    )[0]["summary_text"]

    return {
        "title"        : doc["title"],
        "audience"     : doc["audience"],
        "summary"      : raw_summary,
        "word_count"   : len(raw_summary.split()),
        "generated_at" : datetime.now().strftime("%Y-%m-%d %H:%M"),
    }



In [6]:

# ─────────────────────────────────────────────────────────────
# 5. BATCH PROCESS ALL DESIGN DOCUMENTS
# ─────────────────────────────────────────────────────────────

print("=" * 65)
print("  SDLC DESIGN PHASE — Executive Summary Generator")
print("  Model : facebook/bart-large-cnn")
print("=" * 65)

summaries = []

for doc in design_documents:
    result = generate_executive_summary(doc, summariser, AUDIENCE_CONFIG)
    summaries.append(result)

    print(f"\n{'─' * 65}")
    print(f"  Document : {result['title']}")
    print(f"  Audience : {result['audience']}")
    print(f"  Generated: {result['generated_at']}  |  Words: {result['word_count']}")
    print(f"{'─' * 65}")
    print(f"\n  {result['summary']}\n")



  SDLC DESIGN PHASE — Executive Summary Generator
  Model : facebook/bart-large-cnn

─────────────────────────────────────────────────────────────────
  Document : Authentication Service — Architecture Design
  Audience : CTO / Product Owner
  Generated: 2026-05-06 06:49  |  Words: 53
─────────────────────────────────────────────────────────────────

   authentication service will be implemented as a stateless microservice using OAuth 2.0 with PKCE flow for all public-facing clients. JWTs will be signed with RS256 asymmetric keys rotated every 24 hours via AWS KMS. Token expiry is set to 15 minutes for access tokens and 7 days for refresh tokens stored in cookies.


─────────────────────────────────────────────────────────────────
  Document : Database Schema — Order Management System
  Audience : Project Manager / Business Analyst
  Generated: 2026-05-06 06:49  |  Words: 49
─────────────────────────────────────────────────────────────────

  The order management database will use Post

<h2>Phase 4 — Implementation: Multilingual code comments</h2>

In [2]:
from transformers import pipeline

translator = pipeline("translation",
                      model="Helsinki-NLP/opus-mt-en-fr")   # English → French



In [3]:
# Inline docstring written by an English-speaking developer
english_docstring = """
    Validates the user's JWT token by checking its signature, expiry timestamp,
    and issuer claim. Raises AuthenticationError if the token is invalid or expired.
    Returns the decoded payload dictionary on success.
"""

print("=== Implementation Phase: Docstring Localisation (EN → FR) ===\n")
print("Original (English):")
print(english_docstring.strip())

translated = translator(english_docstring, max_length=256)[0]["translation_text"]
print(f"\nTranslated (French):\n{translated.strip()}")

=== Implementation Phase: Docstring Localisation (EN → FR) ===

Original (English):
Validates the user's JWT token by checking its signature, expiry timestamp,
    and issuer claim. Raises AuthenticationError if the token is invalid or expired.
    Returns the decoded payload dictionary on success.

Translated (French):
Valide le jeton JWT de l'utilisateur en vérifiant sa signature, son timestamp d'expiration et sa réclamation de l'émetteur. Lève AuthentificationErreur si le jeton est invalide ou expiré. Retourne le dictionnaire de charge utile décodé en cas de succès.


<h2>Phase 5 — Testing: Automated bug triage</h2>

In [4]:
#Phase 5 — Testing: Automated bug triage
from transformers import pipeline

analyser = pipeline("sentiment-analysis",
                    model="distilbert-base-uncased-finetuned-sst-2-english")


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [19]:
bug_reports = [
    {"id": "BUG-101", "text": "App crashes every time I open the settings page."},
    {"id": "BUG-102", "text": "The filter works fine and app is working good, but could load a bit faster."},
    {"id": "BUG-103", "text": "Absolutely broken — data gets wiped after logout!"},
    {"id": "BUG-104", "text": "UI is fine , but Minor issue on the profile avatar."},
]

print("=== Testing Phase: Bug Report Triage ===\n")
print(f"{'ID':<10} {'Severity':<12} {'Score':<8} Description")
print("-" * 70)

critical, normal = [], []

for report in bug_reports:
    result   = analyser(report["text"])[0]
    label    = result["label"]
    score    = result["score"]
    severity = "CRITICAL" if label == "NEGATIVE" and score > 0.98 else "NORMAL"

    if severity == "CRITICAL":
        critical.append(report["id"])

    print(f"{report['id']:<10} {severity:<12} {score:.4f}  {report['text'][:45]}...")

print(f"\n{len(critical)} critical bug(s) flagged for immediate escalation: {critical}")

=== Testing Phase: Bug Report Triage ===

ID         Severity     Score    Description
----------------------------------------------------------------------
BUG-101    CRITICAL     0.9996  App crashes every time I open the settings pa...
BUG-102    NORMAL       0.9188  The filter works fine and app is working good...
BUG-103    CRITICAL     0.9998  Absolutely broken — data gets wiped after log...
BUG-104    NORMAL       0.9713  UI is fine , but Minor issue on the profile a...

2 critical bug(s) flagged for immediate escalation: ['BUG-101', 'BUG-103']


<h2>Phase 6 — Deployment: Auto-generated release notes</h2>

In [20]:
#Phase 6 — Deployment: Auto-generated release notes

from transformers import pipeline

summariser = pipeline("summarization",
                      model="facebook/bart-large-cnn",
                      min_length=40,
                      max_length=100)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [21]:
commit_log = """
    - feat: Add CSV export to order history page with date and category filters
    - fix: Resolve crash on settings page when user has no profile photo set
    - perf: Lazy-load product images on homepage reducing LCP by 400ms
    - fix: Correct incorrect delivery date display for international shipments
    - feat: Introduce offline mode with local cache for the last 50 orders
    - chore: Upgrade React from 18.1 to 18.3, update all peer dependencies
    - fix: Prevent data wipe on logout — persist local draft state to storage
"""

print("=== Deployment Phase: Auto Release Notes Generation ===\n")
print("Commit log (raw):")
print(commit_log.strip())

notes = summariser(commit_log)[0]["summary_text"]
print(f"\nGenerated release notes:\n  {notes}")

=== Deployment Phase: Auto Release Notes Generation ===

Commit log (raw):
- feat: Add CSV export to order history page with date and category filters
    - fix: Resolve crash on settings page when user has no profile photo set
    - perf: Lazy-load product images on homepage reducing LCP by 400ms
    - fix: Correct incorrect delivery date display for international shipments
    - feat: Introduce offline mode with local cache for the last 50 orders
    - chore: Upgrade React from 18.1 to 18.3, update all peer dependencies
    - fix: Prevent data wipe on logout — persist local draft state to storage

Generated release notes:
  React has been updated from 18.1 to 18.3, update all peer dependencies and add CSV export to order history page with date and category filters. Lazy-load product images on homepage reducing LCP by 400ms.


<h2>Phase 7 — Maintenance: Localising error messages</h2>

In [22]:
#Phase 7 — Maintenance: Localising error messages

from transformers import pipeline

# Build a multi-language error message localiser
TARGET_LANGUAGES = {
    "French"   : "Helsinki-NLP/opus-mt-en-fr",
    "Spanish"  : "Helsinki-NLP/opus-mt-en-es",
    "German"   : "Helsinki-NLP/opus-mt-en-de",
}

ERROR_MESSAGES = [
    "Your session has expired. Please log in again.",
    "Payment failed. Check your card details and try again.",
    "File upload limit exceeded. Maximum size is 10 MB.",
]

print("=== Maintenance Phase: Error Message Localisation ===\n")

for lang, model_name in TARGET_LANGUAGES.items():
    translator = pipeline("translation", model=model_name)
    print(f"--- {lang} ---")
    for msg in ERROR_MESSAGES:
        result = translator(msg, max_length=128)[0]["translation_text"]
        print(f"  EN: {msg}")
        print(f"  {lang[:2].upper()}: {result}\n")

=== Maintenance Phase: Error Message Localisation ===

--- French ---
  EN: Your session has expired. Please log in again.
  FR: Votre session a expiré. Veuillez vous connecter à nouveau.

  EN: Payment failed. Check your card details and try again.
  FR: Le paiement a échoué. Vérifiez les détails de votre carte et essayez à nouveau.

  EN: File upload limit exceeded. Maximum size is 10 MB.
  FR: La limite de téléchargement des fichiers est dépassée. La taille maximale est de 10 Mo.



config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/768k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/797k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

--- German ---
  EN: Your session has expired. Please log in again.
  GE: Ihre Sitzung ist abgelaufen. Bitte melden Sie sich erneut an.

  EN: Payment failed. Check your card details and try again.
  GE: Zahlung fehlgeschlagen. Überprüfen Sie Ihre Kartendetails und versuchen Sie es erneut.

  EN: File upload limit exceeded. Maximum size is 10 MB.
  GE: Datei-Upload-Grenze überschritten. Maximale Größe ist 10 MB.

